In [1]:
# distilbert_assignment_starter.py

!pip install -q transformers datasets

import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
)
from datasets import load_dataset
from transformers import Trainer, TrainingArguments


In [2]:
# -------------------------
# Device
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
# https://huggingface.co/docs/transformers/en/model_doc/distilbert

# -------------------------
# MLM Task
# -------------------------
def run_mlm_example():
    sentence = "Machine learning is very [MASK]"

    # tokenize sentence and move to device
    inputs = tokenizer(sentence, return_tensors="pt")
    inputs = inputs.to(device)

    # load masked LM model and move to device
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    model = model.to(
        device
    )  # move model to GPU (cuda) or CPU depending on availability

    # run model and get logits
    with torch.no_grad():
        logits = model(**inputs).logits

    # find index of [MASK]
    mask_index = (
        (inputs.input_ids == tokenizer.mask_token_id)[0]
        .nonzero(as_tuple=True)[0]
        .item()
    )

    # compute probabilities and get top predictions
    probability = torch.softmax(
        logits[0, mask_index], dim=0
    )  # compute probabilities from logits convert using softmax
    top = torch.topk(probability, k=10)  # display top predicted words, e.g., 10

    # print reconstructed sentences (replace [MASK])
    for token_id, prob in zip(top.indices, top.values):
        word = tokenizer.decode([token_id.item()])
        reconstructed = sentence.replace("[MASK]", word)
        print(f"{reconstructed}  (prob: {prob*100:.2f}%)")

In [4]:
# -------------------------
# Load dataset
# -------------------------
def load_data():
    dataset = load_dataset("imdb")

    # select subset (500 train, 200 validation)
    train = dataset["train"].shuffle(seed=42).select(range(500))
    validation = dataset["test"].shuffle(seed=42).select(range(200))
    return train, validation

In [5]:
# -------------------------
# Tokenization
# -------------------------
def tokenize_function(example):
    # tokenize with truncation, padding, max_length=64
    inputs = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=64,
    )

    return inputs

In [6]:
# -------------------------
# Training
# -------------------------
def train_model(dataset):
    train, validation = dataset

    # tokenize datasets
    tokenized_train = train.map(tokenize_function)
    tokenized_val = validation.map(tokenize_function)

    # set format for torch
    # tokenized_train.set_format(...)
    # tokenized_val.set_format(...)
    # set format with torch, and specify columns to include input_ids, attention_mask, and label
    # input_ids: tokenized input representation of the text
    # attention_mask: indicates which tokens are padding (0) vs real tokens (1)
    # label: the sentiment label (0 or 1)
    tokenized_train.set_format(
        type="torch", columns=["input_ids", "attention_mask", "label"]
    )
    tokenized_val.set_format(
        type="torch", columns=["input_ids", "attention_mask", "label"]
    )

    # load classification model
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model = model.to(device)

    # IMPORTANT: keep these hyperparameters fixed
    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=1,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        logging_steps=20,
    )

    # define Trainer and train
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
    )
    trainer.train()

    return trainer

In [7]:
# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    run_mlm_example()

    dataset = load_data()
    trainer = train_model(dataset)

    # evaluate model and print accuracy
    _, val = dataset
    tokenized_val = val.map(tokenize_function)
    tokenized_val.set_format(
        type="torch", columns=["input_ids", "attention_mask", "label"]
    )
    predictions = trainer.predict(tokenized_val)
    correct = 0
    for pred, label in zip(predictions.predictions, predictions.label_ids):
        if pred[0] < pred[1]:  # pred[1] > pred[0] means class 1 is predicted
            predicted_class = 1
        else:
            predicted_class = 0
        if predicted_class == label:
            correct += 1

    accuracy = correct / len(predictions.label_ids)
    print(f"Validation Accuracy: {accuracy:.4f}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Machine learning is very important  (prob: 14.64%)
Machine learning is very useful  (prob: 9.30%)
Machine learning is very efficient  (prob: 9.07%)
Machine learning is very .  (prob: 7.30%)
Machine learning is very popular  (prob: 6.68%)
Machine learning is very effective  (prob: 4.19%)
Machine learning is very basic  (prob: 3.82%)
Machine learning is very fast  (prob: 1.98%)
Machine learning is very common  (prob: 1.83%)
Machine learning is very easy  (prob: 1.67%)


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

...

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
20,0.691220


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Validation Accuracy: 0.5850
